In [0]:
%python
# dml/06_carga_analytics_performance_fiis.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("🚀 Iniciando consolidação dos TOP FIIs de cada segmento para a tabela Gold...")

# %%
# 1. Monta a query unificada para capturar o TOP 1 de cada segmento de Tijolo, o TOP 3 de Papel e o TOP 3 de FoF
# Adicionado mapeamento para preencher o novo campo "segmento_alvo" na Gold
qry_consolidacao_gold = f"""
  WITH top_tijolo AS (
    -- Para Tijolo, buscamos o TOP 1 de cada segmento de atuação (garante diversificação setorial absoluta!)
    SELECT 
      s.ticker,
      d.nome_fundo,
      s.preco_atual,
      s.p_vp,
      s.dividend_yield_12m,
      s.score_final,
      s.posicao_ranking,
      s.data_referencia,
      s.segmento_alvo
    FROM {catalogo}.{schema}.stg_scoring_tijolo s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    WHERE s.posicao_ranking = 1 -- Pega o vencedor de cada setor (ex: melhor Shopping, melhor Logística, etc.)
  ),
  
  top_papel AS (
    -- Para Papel, capturamos os 3 fundos de recebíveis com melhor score global
    SELECT 
      s.ticker,
      d.nome_fundo,
      s.preco_atual,
      s.p_vp,
      s.dividend_yield_12m,
      s.score_final,
      s.posicao_ranking,
      s.data_referencia,
      s.segmento_alvo
    FROM {catalogo}.{schema}.stg_scoring_papel s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    WHERE s.posicao_ranking <= 3
  ),
  
  top_fof AS (
    -- Para FoF, capturamos os 3 fundos de fundos com melhor score global
    SELECT 
      s.ticker,
      d.nome_fundo,
      s.preco_atual,
      s.p_vp,
      s.dividend_yield_12m,
      s.score_final,
      s.posicao_ranking,
      s.data_referencia,
      s.segmento_alvo
    FROM {catalogo}.{schema}.stg_scoring_fof s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    WHERE s.posicao_ranking <= 3
  )
  
  -- Unifica os resultados na tabela de consolidação final
  SELECT * FROM top_tijolo
  UNION ALL
  SELECT * FROM top_papel
  UNION ALL
  SELECT * FROM top_fof
"""

df_gold = spark.sql(qry_consolidacao_gold)
df_gold.createOrReplaceTempView("temp_consolidacao_gold")

# %%
# 2. Executa a gravação atômica especificando as colunas exatas de destino
tabela_destino = "analytics_performance_fiis"

# PASSAMOS AS COLUNAS EXPLICITAMENTE NO INSERT OVERWRITE. Isso mapeia por nome e resolve o Column Mapping!
qry_insert_dim_fiis = f"""
  INSERT OVERWRITE {catalogo}.{schema}.{tabela_destino} (
    ticker,
    segmento_alvo,
    nome_fundo,
    preco_atual,
    p_vp,
    dividend_yield_12m,
    score_final,
    posicao_ranking,
    data_selecao,
    data_carga
  )
  SELECT 
    ticker,
    segmento_alvo,
    nome_fundo,
    preco_atual,
    p_vp,
    dividend_yield_12m,
    score_final,
    posicao_ranking,
    data_referencia AS data_selecao,
    CURRENT_TIMESTAMP() AS data_carga
  FROM temp_consolidacao_gold
"""

print(f"Gravando a seleção final unificada na tabela Gold: {catalogo}.{schema}.{tabela_destino}...")

# Grava de forma atômica limpando os dados anteriores da tabela Gold
spark.sql(qry_insert_dim_fiis)

print("✅ Tabela Gold 'analytics_performance_fiis' atualizada com SUCESSO com a seleção diversificada de FIIs!")